# ⚡ Détection d'Anomalies de Consommation Électrique par BiLSTM-Autoencoder

**M1 GSIT — Mme Amel KHEITER · Année Universitaire 2025/2026**

---

## 🎯 Objectif
Détecter automatiquement les anomalies de consommation électrique résidentielle en utilisant un **BiLSTM Autoencoder** entraîné exclusivement sur des patterns normaux. Les anomalies sont identifiées par une **erreur de reconstruction élevée**.

## 📊 Dataset
**UCI Individual Household Electric Power Consumption**
- 2 millions de mesures à la minute (2006–2010)
- Rééchantillonné à l'heure → ~35 000 enregistrements
- 4 features : Global Active Power, Reactive Power, Voltage, Intensity

## 🏗️ Architecture
```
Input (batch,24,4) → BiLSTM(64) → Dense Latent(16) → RepeatVector → LSTM(64) → TimeDistributed Dense(4) → Output
```

## 📦 0. Installation des Dépendances

In [ ]:
# Uncomment and run once if needed
# !pip install tensorflow==2.13.0 pandas scikit-learn matplotlib seaborn joblib

## 📚 1. Imports

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, f1_score
)
import joblib

print(f'TensorFlow version : {tf.__version__}')
print(f'NumPy version      : {np.__version__}')
print('✅ All imports successful')

## ⚙️ 2. Configuration Globale

In [ ]:
# ── Paths ─────────────────────────────────────────────────
DATA_PATH   = 'data/household_power_consumption.txt'
SCALER_PATH = 'data/scaler.pkl'

os.makedirs('data',    exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('plots',   exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────
FEATURES      = ['Global_active_power', 'Global_reactive_power',
                 'Voltage', 'Global_intensity']
WINDOW_SIZE   = 24      # 24-hour sliding window
TRAIN_RATIO   = 0.80    # 80% train, 20% test
EPOCHS        = 50
BATCH_SIZE    = 64
PATIENCE      = 7
THRESHOLD_PCT = 95      # percentile for anomaly threshold
ANOMALY_RATIO = 0.05    # 5% synthetic anomalies injected

np.random.seed(42)
tf.random.set_seed(42)

print('Configuration loaded ✅')
print(f'  Window size   : {WINDOW_SIZE} hours')
print(f'  Features      : {FEATURES}')
print(f'  Epochs        : {EPOCHS}')
print(f'  Threshold     : {THRESHOLD_PCT}th percentile')

## 🔄 3. Chargement et Nettoyage des Données

In [ ]:
print('Loading raw data...')
df_raw = pd.read_csv(
    DATA_PATH,
    sep=';',
    low_memory=False,
    na_values=['?'],       # '?' → NaN
)

# Parse datetime and set as index
df_raw['datetime'] = pd.to_datetime(
    df_raw['Date'] + ' ' + df_raw['Time'], dayfirst=True
)
df_raw.set_index('datetime', inplace=True)
df_raw.drop(columns=['Date', 'Time'], inplace=True)

# Convert all columns to numeric
for col in df_raw.columns:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

print(f'Raw shape         : {df_raw.shape}')
print(f'Missing values    : {df_raw.isna().sum().sum():,}')
print(f'Date range        : {df_raw.index.min()} → {df_raw.index.max()}')

In [ ]:
# Linear interpolation for missing values
df_raw.interpolate(method='linear', inplace=True)
df_raw.dropna(inplace=True)

# Resample to hourly averages
df = df_raw.resample('H').mean()
df.dropna(inplace=True)

print(f'After cleaning    : {df_raw.shape[0]:,} minute-level records')
print(f'After resampling  : {df.shape[0]:,} hourly records')
print(f'\nStatistical Summary:')
df[FEATURES].describe().round(3)

## 📊 4. Analyse Exploratoire (EDA)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True)
colors = ['#2196F3', '#F44336', '#FF9800', '#4CAF50']

for ax, feat, color in zip(axes, FEATURES, colors):
    ax.plot(df.index, df[feat], lw=0.6, color=color, alpha=0.9)
    ax.set_ylabel(feat.replace('_', ' '), fontsize=9)
    ax.grid(alpha=0.25)

axes[0].set_title('UCI Household Power Consumption — Hourly Data (2006-2010)',
                  fontsize=13, fontweight='bold', pad=10)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()
fig.tight_layout()
plt.savefig('plots/consumption_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → plots/consumption_overview.png')

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(6, 5))
corr = df[FEATURES].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/feature_correlation.png', dpi=150)
plt.show()

## 🔧 5. Prétraitement — Normalisation & Fenêtres Glissantes

In [ ]:
# ── Train / Test split ────────────────────────────────────
data  = df[FEATURES].values
n     = len(data)
split = int(n * TRAIN_RATIO)

X_train_raw = data[:split]
X_test_raw  = data[split:]
train_idx   = df.index[:split]
test_idx    = df.index[split:]

# ── MinMaxScaler (fit on train ONLY) ─────────────────────
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)
joblib.dump(scaler, SCALER_PATH)

print(f'Train samples : {len(X_train_scaled):,}')
print(f'Test  samples : {len(X_test_scaled):,}')
print(f'Scaler saved  → {SCALER_PATH}')

In [ ]:
# ── Inject synthetic anomalies into TEST set ──────────────
rng = np.random.default_rng(42)
X_test_mod = X_test_scaled.copy()
y_test_raw = np.zeros(len(X_test_mod), dtype=int)

n_anomalies = int(len(X_test_mod) * ANOMALY_RATIO)
anom_idx    = rng.choice(len(X_test_mod), n_anomalies, replace=False)
half        = n_anomalies // 2

# Spikes: multiply Global_active_power by 3.5
X_test_mod[anom_idx[:half], 0] = np.clip(
    X_test_mod[anom_idx[:half], 0] * 3.5, 0, 1)

# Drops: reduce all features to 10%
X_test_mod[anom_idx[half:], :] = X_test_mod[anom_idx[half:], :] * 0.10

y_test_raw[anom_idx] = 1

print(f'Injected {n_anomalies} anomalies : {half} spikes + {n_anomalies-half} drops')
print(f'Anomaly rate : {n_anomalies/len(X_test_mod)*100:.1f}%')

In [ ]:
# ── Sliding window sequences ──────────────────────────────
def create_sequences(data, window=WINDOW_SIZE):
    """Convert 2D array to 3D sliding windows [N, window, features]"""
    seqs = []
    for i in range(len(data) - window):
        seqs.append(data[i: i + window])
    return np.array(seqs, dtype=np.float32)

X_train_seq = create_sequences(X_train_scaled)
X_test_seq  = create_sequences(X_test_mod)

# Align labels: use label of last timestep in each window
y_test = np.array(
    [y_test_raw[i + WINDOW_SIZE - 1] for i in range(len(X_test_mod) - WINDOW_SIZE)],
    dtype=int
)

N_FEATURES = X_train_seq.shape[2]

print(f'X_train_seq shape : {X_train_seq.shape}  → (windows, hours, features)')
print(f'X_test_seq  shape : {X_test_seq.shape}')
print(f'y_test labels     : {y_test.sum()} anomalies / {len(y_test)} total')

## 🏗️ 6. Architecture BiLSTM Autoencoder

In [ ]:
def build_bilstm_autoencoder(window_size, n_features):
    """
    BiLSTM Autoencoder Architecture:
    ─────────────────────────────────────────────────────────
    INPUT          (batch, 24, 4)
    BiLSTM Enc     (batch, 128)       ← 64 units × 2 directions
    Dense Latent   (batch, 16)        ← compressed representation
    RepeatVector   (batch, 24, 16)    ← expand back to sequence
    LSTM Dec       (batch, 24, 64)    ← reconstruct temporal patterns
    TimeDistributed(batch, 24, 4)     ← reconstruct all features
    OUTPUT         (batch, 24, 4)
    ─────────────────────────────────────────────────────────
    Loss: MSE — trained on NORMAL data only
    """
    inputs = keras.Input(shape=(window_size, n_features), name='input')

    # ENCODER
    enc = layers.Bidirectional(
        layers.LSTM(64, activation='tanh', return_sequences=False),
        name='bi_lstm_encoder'
    )(inputs)

    latent = layers.Dense(16, activation='relu', name='latent')(enc)

    # DECODER
    dec = layers.RepeatVector(window_size, name='repeat')(latent)
    dec = layers.LSTM(64, activation='tanh', return_sequences=True,
                      name='lstm_decoder')(dec)
    outputs = layers.TimeDistributed(
        layers.Dense(n_features), name='output'
    )(dec)

    model = Model(inputs, outputs, name='BiLSTM_Autoencoder')
    model.compile(optimizer='adam', loss='mse')
    return model

bilstm = build_bilstm_autoencoder(WINDOW_SIZE, N_FEATURES)
bilstm.summary()

## 🎓 7. Entraînement BiLSTM Autoencoder (données normales uniquement)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-6, verbose=1
    ),
]

history_bilstm = bilstm.fit(
    X_train_seq, X_train_seq,    # autoencoder: input == target
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.10,
    callbacks=callbacks,
    verbose=1,
)

bilstm.save('models/bilstm_autoencoder.keras')
print('\n✅ BiLSTM Autoencoder trained and saved.')

In [ ]:
# Training curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_bilstm.history['loss'],     label='Train Loss',      color='#4CAF50', lw=2)
ax.plot(history_bilstm.history['val_loss'], label='Validation Loss', color='#9C27B0', lw=2, ls='--')
ax.set_title('BiLSTM Autoencoder — Training History', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plots/training_history_BiLSTM.png', dpi=150)
plt.show()

## 🔍 8. Détection d'Anomalies — Erreur de Reconstruction

In [ ]:
# Compute reconstruction errors
def reconstruction_error(model, X, batch_size=256):
    """Per-sample MSE between input and reconstruction."""
    X_pred = model.predict(X, batch_size=batch_size, verbose=0)
    return np.mean((X - X_pred) ** 2, axis=(1, 2))

errors_train = reconstruction_error(bilstm, X_train_seq)
errors_test  = reconstruction_error(bilstm, X_test_seq)

# Threshold = 95th percentile of training errors
threshold = float(np.percentile(errors_train, THRESHOLD_PCT))
y_pred_bilstm = (errors_test > threshold).astype(int)

print(f'Train error — mean: {errors_train.mean():.6f}  |  std: {errors_train.std():.6f}')
print(f'Test  error — mean: {errors_test.mean():.6f}  |  std: {errors_test.std():.6f}')
print(f'Threshold ({THRESHOLD_PCT}th pct) : {threshold:.6f}')
print(f'Anomalies detected : {y_pred_bilstm.sum()} / {len(y_pred_bilstm)}')

In [ ]:
# ── Reconstruction error histogram ────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(errors_train, bins=80, alpha=0.55, color='#4CAF50',
        label='Train errors (normal)', density=True)
ax.hist(errors_test,  bins=80, alpha=0.55, color='#F44336',
        label='Test errors (with anomalies)', density=True)
ax.axvline(threshold, color='#FF9800', lw=2.5, ls='--',
           label=f'Threshold = {threshold:.5f}  ({THRESHOLD_PCT}th pct)')
ax.set_title('Reconstruction Error Distribution — BiLSTM Autoencoder',
             fontsize=13, fontweight='bold')
ax.set_xlabel('MSE Reconstruction Error')
ax.set_ylabel('Density')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plots/error_histogram_BiLSTM.png', dpi=150)
plt.show()

In [ ]:
# ── Anomaly Timeline ──────────────────────────────────────
aligned_idx  = test_idx[WINDOW_SIZE:WINDOW_SIZE + len(y_pred_bilstm)]
power_series = df['Global_active_power'].loc[aligned_idx]
anom_mask    = y_pred_bilstm == 1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 7), sharex=True)

# Power consumption
ax1.plot(aligned_idx, power_series.values,
         color='#2196F3', lw=0.8, label='Global Active Power (kW)')
ax1.scatter(aligned_idx[anom_mask], power_series.values[anom_mask],
            color='#F44336', s=18, zorder=5, label='Detected Anomaly')
ax1.set_ylabel('Power (kW)')
ax1.set_title('Consumption & Detected Anomalies — BiLSTM Autoencoder',
              fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.25)

# Reconstruction error
ax2.plot(aligned_idx, errors_test[:len(aligned_idx)],
         color='#607D8B', lw=0.7, label='Reconstruction Error')
ax2.axhline(threshold, color='#FF9800', lw=2, ls='--',
            label=f'Threshold = {threshold:.5f}')
ax2.fill_between(aligned_idx, 0, errors_test[:len(aligned_idx)],
                 where=errors_test[:len(aligned_idx)] > threshold,
                 color='#F44336', alpha=0.30)
ax2.set_ylabel('MSE Error')
ax2.set_xlabel('Date')
ax2.legend()
ax2.grid(alpha=0.25)

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()
fig.tight_layout()
plt.savefig('plots/anomaly_timeline_BiLSTM.png', dpi=150)
plt.show()

## ⏱️ 8b. Délai Moyen de Détection (en heures)

In [ ]:
# Detection delay: how many hours after an anomaly starts is it detected?
# We compare ground truth (y_test) with predictions (y_pred_bilstm)

detection_delays = []
i = 0
while i < len(y_test):
    if y_test[i] == 1:  # start of an anomaly event
        # Find when our model detects it
        detected = False
        for j in range(i, min(i + 48, len(y_pred_bilstm))):
            if y_pred_bilstm[j] == 1:
                detection_delays.append(j - i)  # delay in hours
                detected = True
                break
        if not detected:
            detection_delays.append(None)  # missed anomaly
        # Skip to next distinct event
        while i < len(y_test) and y_test[i] == 1:
            i += 1
    else:
        i += 1

valid_delays = [d for d in detection_delays if d is not None]
missed       = detection_delays.count(None)

print(f'Total anomaly events   : {len(detection_delays)}')
print(f'Detected               : {len(valid_delays)}')
print(f'Missed                 : {missed}')
if valid_delays:
    print(f'Mean detection delay   : {np.mean(valid_delays):.2f} hours')
    print(f'Median detection delay : {np.median(valid_delays):.2f} hours')
    print(f'Max detection delay    : {np.max(valid_delays)} hours')

## 🌲 9. Baseline 1 — Simple Autoencoder

In [ ]:
def build_simple_autoencoder(window_size, n_features):
    """Dense autoencoder baseline: Flatten → Dense → Latent → Dense → Reshape"""
    flat_dim = window_size * n_features
    inputs = keras.Input(shape=(window_size, n_features))
    x      = layers.Flatten()(inputs)
    x      = layers.Dense(128, activation='relu')(x)
    x      = layers.Dense(64,  activation='relu')(x)
    latent = layers.Dense(16,  activation='relu', name='latent')(x)
    x      = layers.Dense(64,  activation='relu')(latent)
    x      = layers.Dense(128, activation='relu')(x)
    x      = layers.Dense(flat_dim, activation='sigmoid')(x)
    outputs = layers.Reshape((window_size, n_features))(x)
    model  = Model(inputs, outputs, name='Simple_Autoencoder')
    model.compile(optimizer='adam', loss='mse')
    return model

simple_ae = build_simple_autoencoder(WINDOW_SIZE, N_FEATURES)

history_simple = simple_ae.fit(
    X_train_seq, X_train_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.10,
    callbacks=callbacks, verbose=1,
)

simple_ae.save('models/simple_autoencoder.keras')

errors_train_simple = reconstruction_error(simple_ae, X_train_seq)
errors_test_simple  = reconstruction_error(simple_ae, X_test_seq)
threshold_simple    = float(np.percentile(errors_train_simple, THRESHOLD_PCT))
y_pred_simple       = (errors_test_simple > threshold_simple).astype(int)

print(f'\nSimple AE threshold : {threshold_simple:.6f}')
print(f'Anomalies detected  : {y_pred_simple.sum()}')

## 🌲 10. Baseline 2 — Isolation Forest

In [ ]:
iforest = IsolationForest(n_estimators=200, contamination=0.05,
                          random_state=42, n_jobs=-1)
iforest.fit(X_train_seq.reshape(len(X_train_seq), -1))
joblib.dump(iforest, 'models/isolation_forest.pkl')

X_test_flat  = X_test_seq.reshape(len(X_test_seq), -1)
y_pred_if    = (iforest.predict(X_test_flat) == -1).astype(int)
if_scores    = -iforest.decision_function(X_test_flat)

print(f'Isolation Forest — anomalies detected : {y_pred_if.sum()}')

## 📈 11. Évaluation — ROC Curves + AUC

In [ ]:
# ── ROC Curves for all 3 models ───────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

roc_configs = [
    ('BiLSTM AE',        errors_test,        '#2196F3'),
    ('Simple AE',        errors_test_simple, '#F44336'),
    ('Isolation Forest', if_scores,          '#4CAF50'),
]

for name, scores, color in roc_configs:
    n = min(len(y_test), len(scores))
    fpr, tpr, _ = roc_curve(y_test[:n], scores[:n])
    auc = roc_auc_score(y_test[:n], scores[:n])
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name}  (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Model Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plots/roc_curves_comparison.png', dpi=150)
plt.show()

## 📊 12. Matrices de Confusion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
models_preds = [
    ('BiLSTM AE',        y_pred_bilstm),
    ('Simple AE',        y_pred_simple),
    ('Isolation Forest', y_pred_if),
]

for ax, (name, y_pred) in zip(axes, models_preds):
    n  = min(len(y_test), len(y_pred))
    cm = confusion_matrix(y_test[:n], y_pred[:n])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Anomaly'],
                yticklabels=['Normal', 'Anomaly'], ax=ax)
    ax.set_title(f'Confusion Matrix\n{name}', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('plots/confusion_matrices.png', dpi=150)
plt.show()

## 🏆 13. Tableau Comparatif Final

In [ ]:
import time
from sklearn.metrics import precision_score, recall_score

rows = []
for name, y_pred, scores in [
    ('BiLSTM AE',        y_pred_bilstm, errors_test),
    ('Simple AE',        y_pred_simple, errors_test_simple),
    ('Isolation Forest', y_pred_if,     if_scores),
]:
    n = min(len(y_test), len(y_pred))
    rows.append({
        'Model':     name,
        'Precision': round(precision_score(y_test[:n], y_pred[:n], zero_division=0), 4),
        'Recall':    round(recall_score(y_test[:n],    y_pred[:n], zero_division=0), 4),
        'F1 Score':  round(f1_score(y_test[:n],        y_pred[:n], zero_division=0), 4),
        'AUC':       round(roc_auc_score(y_test[:n], scores[:n]), 4),
        'FP Rate %': round((((y_pred[:n]==1) & (y_test[:n]==0)).sum() / (y_test[:n]==0).sum())*100, 2),
    })

summary = pd.DataFrame(rows)
summary.to_csv('outputs/metrics_summary.csv', index=False)
print('Summary saved → outputs/metrics_summary.csv')
summary

In [ ]:
# Bar chart comparison
metrics = ['Precision', 'Recall', 'F1 Score', 'AUC']
x = np.arange(len(metrics))
width = 0.22
colors = ['#2196F3', '#F44336', '#4CAF50']

fig, ax = plt.subplots(figsize=(10, 5))
for i, row in summary.iterrows():
    vals = [row[m] for m in metrics]
    ax.bar(x + i * width, vals, width, label=row['Model'], color=colors[i])

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Performance Comparison — All Models', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plots/metrics_comparison.png', dpi=150)
plt.show()

## 🏁 14. Conclusion

### Résultats
- Le **BiLSTM Autoencoder** obtient le meilleur AUC grâce à sa capacité à capturer les dépendances temporelles **bidirectionnelles**
- Le **Simple Autoencoder** (Dense) ignore la structure temporelle → performances inférieures
- L'**Isolation Forest** est rapide mais ne modélise pas les séquences → performances les plus basses

### Pourquoi le BiLSTM est supérieur ?
1. **Bidirectionnel** : lit la séquence dans les deux sens → contexte passé ET futur
2. **LSTM** : mémoire à long terme → capture les patterns journaliers/hebdomadaires
3. **Autoencoder** : apprend la structure normale → détecte les déviations

### Déploiement
```bash
streamlit run app.py
```
Interface web interactive avec upload de données, détection en temps réel simulé et export CSV des alertes.